# Reviewer 4 (round 2), Comment 2 — Hyperparameter-selection sensitivity

Reviewer 4 notes that hyperparameters were chosen on a single train/validation split drawn before the k=10 cross-validation, so the CV folds do not give an unbiased estimate after model selection.

This notebook checks whether that selection step can affect the paper's conclusions:

1. **Effective size of the search space.** The grid was `criterion ∈ {gini, entropy} × max_features ∈ {sqrt, log2}` for Random Forest and `max_features ∈ {sqrt, log2}` for Gradient Boosting. With 15 (Exp. A) or 16 (Exp. B) predictors, scikit-learn turns both `sqrt` and `log2` into the same integer (3 and 4 features per split), so the two values give identical models. The search therefore chose between **two** Random Forest configurations and **one** Gradient Boosting configuration.
2. **Every configuration under the same k=10 folds.** Both Random Forest criteria are evaluated in both experiments with the paper's protocol (stratified k=10, `shuffle=True, random_state=42`, RandomOverSampler inside the training fold, `average='macro'`). If the *worse* Random Forest configuration still beats Gradient Boosting in every fold, the comparison does not depend on the selection step.

Gradient Boosting is retrained only for Experiment B (≈2 min). For Experiment A its per-fold F1-macro is taken from `PROCESO_MODELADO_CV_EN_perclass_100926.ipynb` (same folds, same metric), because retraining it takes about 170 minutes and its single configuration cannot change.

Expected runtime: about 25–30 minutes. Results are saved to `reviewer4_hparam_results.json`.

In [ ]:
# ── COLAB SETUP ─────────────────────────────────────────────────────────
# Step 1: mount Google Drive
from google.colab import drive
import os
# If already mounted, this line detects it automatically
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

# Step 2: install libraries not included in Colab
!pip install imbalanced-learn lime -q

# Step 3: copy the parquet file from Drive to the local Colab environment
# ─── ADJUST this path to the folder where you uploaded the parquet in your Drive ───
DRIVE_PARQUET = '/content/drive/MyDrive/fci-datos/part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet'
LOCAL_PARQUET = '/content/part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet'
# ────────────────────────────────────────────────────────────────────────────

if not os.path.exists(LOCAL_PARQUET):
    import shutil
    shutil.copy(DRIVE_PARQUET, LOCAL_PARQUET)
    print(f'Parquet copied to {LOCAL_PARQUET}')
else:
    print(f'Parquet already available at {LOCAL_PARQUET}')

print('Setup complete.')

In [ ]:
import pandas as pd, numpy as np, time, json, warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import clone
from imblearn.over_sampling import RandomOverSampler
from scipy.stats import wilcoxon

df = pd.read_parquet(LOCAL_PARQUET)
for col in df.select_dtypes(include='string').columns:
    df[col] = df[col].astype(object)
for col in ['NUMBER_AWARDS', 'LOTS_NUMBER', 'NUMBER_OFFERS', 'NUMBER_TENDERS_SME']:
    df[col] = df[col].astype(int)
df['GROUP_CPV'] = df['CPV'].astype(str).str[:2]
df['SME_WIN'] = df['B_CONTRACTOR_SME'].astype(str).str.contains('y', case=False, na=False).astype(int)
cat_cols = ['B_MULTIPLE_CAE', 'B_ON_BEHALF', 'CAE_TYPE', 'MAIN_ACTIVITY',
            'ISO_COUNTRY_CODE', 'TYPE_OF_CONTRACT', 'GROUP_CPV']
for col in cat_cols:
    df[col] = df[col].astype(str)
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_arr = ohe.fit_transform(df[cat_cols])
num_cols = ['NUMBER_AWARDS', 'LOTS_NUMBER', 'NUMBER_OFFERS', 'NUMBER_TENDERS_SME']
df_proc = pd.concat([df[num_cols].reset_index(drop=True),
                     pd.DataFrame(ohe_arr, columns=ohe.get_feature_names_out(cat_cols))], axis=1)

used_features_A = ['B_MULTIPLE_CAE_n', 'B_ON_BEHALF_n', 'GROUP_CPV_45', 'GROUP_CPV_33', 'GROUP_CPV_15',
                   'TYPE_OF_CONTRACT_w', 'ISO_COUNTRY_CODE_si', 'NUMBER_AWARDS', 'LOTS_NUMBER',
                   'NUMBER_OFFERS', 'NUMBER_TENDERS_SME', 'CAE_TYPE_3', 'CAE_TYPE_4', 'CAE_TYPE_5',
                   'ISO_COUNTRY_CODE_lu']
used_features_B2 = ['B_MULTIPLE_CAE_n', 'B_ON_BEHALF_n', 'GROUP_CPV_45', 'GROUP_CPV_33', 'GROUP_CPV_15',
                    'TYPE_OF_CONTRACT_w', 'MAIN_ACTIVITY_health', 'ISO_COUNTRY_CODE_si',
                    'NUMBER_AWARDS', 'LOTS_NUMBER', 'NUMBER_OFFERS',
                    'CAE_TYPE_3', 'MAIN_ACTIVITY_general public\\services',
                    'CAE_TYPE_4', 'CAE_TYPE_5', 'ISO_COUNTRY_CODE_lu']
X_A = df_proc[used_features_A].astype(float).to_numpy()
y_A = np.array(df['MAIN_ACTIVITY'].astype(str).tolist())
X_B = df_proc[used_features_B2].astype(float).to_numpy()
y_B = df['SME_WIN'].to_numpy()
print('X_A', X_A.shape, '| X_B', X_B.shape)
RESULTS = {}

## 1. `sqrt` and `log2` give the same number of features per split

In [ ]:
eff = {}
for name, X in [('A', X_A), ('B', X_B)]:
    n = X.shape[1]
    for mf in ['sqrt', 'log2']:
        m = RandomForestClassifier(n_estimators=1, max_features=mf, random_state=42).fit(X[:2000], (np.arange(2000) % 2))
        eff[f'{name}_{mf}'] = int(m.estimators_[0].max_features_)
    print(f'Exp {name}: {n} predictors -> sqrt: {eff[name+"_sqrt"]}, log2: {eff[name+"_log2"]} features per split')
RESULTS['effective_max_features'] = eff
assert eff['A_sqrt'] == eff['A_log2'] and eff['B_sqrt'] == eff['B_log2']
print('-> the max_features dimension of the grid is degenerate: the search had 2 RF and 1 GB effective configurations.')

## 2. Both Random Forest configurations (and Gradient Boosting for Exp. B) under the paper's k=10 folds

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
ros = RandomOverSampler(random_state=42)

def cv_f1(X, y, model, label):
    t0 = time.time()
    scores = []
    for tr, te in skf.split(X, y):
        m = clone(model)
        Xb, yb = ros.fit_resample(X[tr], y[tr])
        m.fit(Xb, yb)
        scores.append(float(f1_score(y[te], m.predict(X[te]), average='macro', zero_division=0)))
    print(f'{label:<28} F1-macro = {np.mean(scores):.4f} ± {np.std(scores):.4f}   ({(time.time()-t0)/60:.1f} min)')
    return scores

def rf(criterion):
    return RandomForestClassifier(criterion=criterion, max_features='sqrt', bootstrap=True, max_samples=2/3,
                                  n_estimators=100, n_jobs=-1, random_state=42)
gb = GradientBoostingClassifier(max_features='sqrt', learning_rate=0.1, n_estimators=100, random_state=42)

folds = {}
folds['B_rf_gini'] = cv_f1(X_B, y_B, rf('gini'), 'Exp B  RF gini')
folds['B_rf_entropy'] = cv_f1(X_B, y_B, rf('entropy'), 'Exp B  RF entropy (paper)')
folds['B_gb'] = cv_f1(X_B, y_B, gb, 'Exp B  GB (paper)')
folds['A_rf_gini'] = cv_f1(X_A, y_A, rf('gini'), 'Exp A  RF gini (paper)')
folds['A_rf_entropy'] = cv_f1(X_A, y_A, rf('entropy'), 'Exp A  RF entropy')
# Gradient Boosting, Experiment A: per-fold F1-macro from PROCESO_MODELADO_CV_EN_perclass_100926.ipynb (cell 27)
folds['A_gb'] = [0.2965, 0.2925, 0.3017, 0.2775, 0.3020, 0.2918, 0.3043, 0.2987, 0.3098, 0.2849]
print(f'{"Exp A  GB (paper, stored)":<28} F1-macro = {np.mean(folds["A_gb"]):.4f} ± {np.std(folds["A_gb"]):.4f}')

for k, ref in [('B_rf_entropy', 0.5863), ('B_gb', 0.5569), ('A_rf_gini', 0.3550)]:
    if abs(np.mean(folds[k]) - ref) > 0.002:
        print(f'WARNING: {k} = {np.mean(folds[k]):.4f} does not reproduce the paper value {ref}')
RESULTS['per_fold'] = folds

## 3. Does the worse Random Forest configuration still beat Gradient Boosting?

In [ ]:
summary = {}
for exp in ['A', 'B']:
    rf_cfgs = {c: np.array(folds[f'{exp}_rf_{c}']) for c in ['gini', 'entropy']}
    g = np.array(folds[f'{exp}_gb'])
    means = {c: float(v.mean()) for c, v in rf_cfgs.items()}
    worst = min(means, key=means.get); best = max(means, key=means.get)
    d = rf_cfgs[worst] - g
    w1 = wilcoxon(rf_cfgs[worst], g, alternative='greater')
    summary[exp] = {'rf_means': means, 'gb_mean': float(g.mean()),
                    'best_rf': best, 'worst_rf': worst,
                    'selection_gap': means[best] - means[worst],
                    'worst_rf_minus_gb_mean': float(d.mean()), 'worst_rf_minus_gb_min': float(d.min()),
                    'worst_rf_wins': int((d > 0).sum()), 'wilcoxon_p_one_sided': float(w1.pvalue)}
    s = summary[exp]
    print(f'Exp {exp}: RF gini={means["gini"]:.4f}, entropy={means["entropy"]:.4f} (gap {s["selection_gap"]:.4f}); GB={s["gb_mean"]:.4f}')
    print(f'        worse RF ({worst}) − GB = {s["worst_rf_minus_gb_mean"]:+.4f} (min fold {s["worst_rf_minus_gb_min"]:+.4f}), '
          f'wins {s["worst_rf_wins"]}/10, one-sided Wilcoxon p = {s["wilcoxon_p_one_sided"]:.4f}')
RESULTS['summary'] = summary

## 4. Save

In [ ]:
with open('reviewer4_hparam_results.json', 'w') as fh:
    json.dump(RESULTS, fh, indent=2)
print('Saved reviewer4_hparam_results.json')
try:
    from google.colab import files
    files.download('reviewer4_hparam_results.json')
except Exception as e:
    print('Download it from the Files panel:', e)